# 9. Paper 1 model tables and plots

Assemble publication outputs for the stacked conditional-logit model and the XGBoost/neural winners frozen in Stage 8. Stage 9 validates the SHA-256 hashes of the development-CV summary, experiment plan, and frozen ranking before it opens any held-out artifact. It never reselects a model from held-out performance.

Primary outputs use `all`. Set `COMMENTGAP_MODEL_SCOPES=all,root` to add matching appendix tables. The reporter refuses to run while the factorial launcher is active.

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
from IPython.display import display

from commentgap_analysis.paper1_reporting import run_paper1_reporting
from commentgap_analysis.paper1_plotting import (
    plot_regression_selector_coefficients,
    plot_regression_selector_differences,
    plot_winner_permutation_importance,
    plot_winner_permutation_importance_gaps,
    plot_winner_shap_importance,
    plot_winner_shap_importance_gaps,
)

scope_text = os.getenv("COMMENTGAP_MODEL_SCOPES", "all")
SCOPES = tuple(dict.fromkeys(part.strip() for part in scope_text.split(",") if part.strip()))
if not SCOPES or not set(SCOPES).issubset({"all", "root"}):
    raise ValueError(f"Invalid COMMENTGAP_MODEL_SCOPES={scope_text!r}")
MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
FACTORIAL_ROOT = Path(os.getenv("COMMENTGAP_FACTORIAL_ROOT", "model_output/selection_2025/factorial_rankers"))
WINNER_ROOT = Path(os.getenv("COMMENTGAP_FACTORIAL_WINNER_ROOT", "model_output/selection_2025/paper1/factorial_winners"))
REGRESSION_ROOT = Path(os.getenv("COMMENTGAP_REGRESSION_ROOT", "model_output/selection_2025/regression"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_REPORT_ROOT", "model_output/selection_2025/paper1/reporting"))
TABLES = OUTPUT_ROOT / "tables"
PERMUTATION_REPEATS = int(os.getenv("COMMENTGAP_PERMUTATION_REPEATS", "1"))
SHAP_TEST_ROWS = int(os.getenv("COMMENTGAP_SHAP_TEST_ROWS", "50000"))
SHAP_BACKGROUND_ROWS = int(os.getenv("COMMENTGAP_SHAP_BACKGROUND_ROWS", "2048"))
SHAP_NSAMPLES = int(os.getenv("COMMENTGAP_SHAP_NSAMPLES", "100"))
SHAP_FORCE_RECOMPUTE = os.getenv("COMMENTGAP_SHAP_FORCE_RECOMPUTE", "0") == "1"
SHAP_CHUNK_ROWS = int(os.getenv("COMMENTGAP_SHAP_CHUNK_ROWS", "500"))
{"scopes": SCOPES, "winner_manifest": str(WINNER_ROOT / "factorial_winner_manifest.json"), "output": str(OUTPUT_ROOT)}

In [ ]:
report_manifest = run_paper1_reporting(
    model_data_root=MODEL_DATA_ROOT,
    factorial_root=FACTORIAL_ROOT,
    winner_root=WINNER_ROOT,
    regression_root=REGRESSION_ROOT,
    output_root=OUTPUT_ROOT,
    scopes=SCOPES,
    bootstrap_draws=1000,
    permutation_repeats=PERMUTATION_REPEATS,
    shap_test_rows=SHAP_TEST_ROWS,
    shap_background_rows=SHAP_BACKGROUND_ROWS,
    shap_nsamples=SHAP_NSAMPLES,
    shap_force_recompute=SHAP_FORCE_RECOMPUTE,
    shap_chunk_rows=SHAP_CHUNK_ROWS,
    make_figures=True,
)
report_manifest

## Selector feature gaps

This section compares three complementary views of feature use: stacked-model coefficients, held-out permutation importance, and SHAP attribution. The regression feature gap is the signed curator-minus-audience interaction on the log-odds scale. XGBoost and neural feature gaps are paired within-article permutation losses: curator nDCG@k importance minus audience nDCG@k importance. SHAP plots use signed mean attribution and signed editor-minus-audience gaps. Positive values indicate an upward contribution or greater editor-specific contribution; negative values indicate a downward contribution or greater audience-specific contribution. Audience importance averages the ten deterministic tie draws. Permutations cover the named tabular features; frozen BGE vectors remain fixed rather than treating their 1,024 latent dimensions as separate substantive features.

In [ ]:
regression_feature_gaps = pd.read_csv(TABLES / "regression_feature_gaps.csv")
permutation_gaps = pd.read_csv(TABLES / "held_out_permutation_importance_gaps.csv")
display(regression_feature_gaps)
display(permutation_gaps.sort_values("permutation_importance_gap", key=abs, ascending=False))
display(plot_regression_selector_coefficients(regression_feature_gaps, output_root=None, show=False))
display(plot_regression_selector_differences(regression_feature_gaps, output_root=None, show=False))
display(plot_winner_permutation_importance(permutation_gaps, output_root=None, show=False))
display(plot_winner_permutation_importance_gaps(permutation_gaps, output_root=None, show=False))

## SHAP artifacts

Stage 9 calculates SHAP for the frozen XGBoost and neural winners on the sealed `paper2_test` comments. The background is development-only; BGE dimensions are summed into the `text_bge` block. The report exports both selector-specific signed mean SHAP contributions and signed editor-minus-audience SHAP gaps; mean absolute SHAP remains available in the table as an importance diagnostic. Notebook 12 consumes the story-aggregated summary for the FORUM comparison.

In [ ]:
shap_summary = pd.read_parquet(TABLES / "held_out_shap_importance.parquet")
display(shap_summary.head(20))
print(f"{len(shap_summary):,} story-aggregated SHAP rows")
display(plot_winner_shap_importance(shap_summary, output_root=None, show=False))
display(plot_winner_shap_importance_gaps(shap_summary, output_root=None, show=False))

## Frozen winners and development CV

In [ ]:
TABLES = OUTPUT_ROOT / "tables"
winners = pd.read_csv(TABLES / "development_cv_winners.csv")
cv_ranking = pd.read_csv(TABLES / "development_cv_variant_ranking.csv")
display(winners)
display(cv_ranking.groupby(["family", "scope"], group_keys=False).head(10))

## Held-out results and paired comparisons

These results are opened only after winner freezing. Paired differences are clustered by held-out article and use the second-minus-first direction named in `comparison`. For `mean_selected_rank`, lower values are better; the other reported ranking metrics are better when higher.

In [ ]:
performance = pd.read_csv(TABLES / "held_out_model_performance.csv")
paired = pd.read_csv(TABLES / "held_out_paired_model_differences.csv")
display(performance)
display(paired[paired["metric"] == "ndcg_at_k"] )

## Regression associations and tie sensitivity

In [ ]:
associations = pd.read_csv(TABLES / "regression_selector_associations.csv")
tie_sensitivity = pd.read_csv(TABLES / "held_out_tie_sensitivity.csv")
display(associations)
display(tie_sensitivity)
json.loads((OUTPUT_ROOT / "report_manifest.json").read_text())